# Tutorial 02: Multi-Turn Conversations - Stateful Travel Assistant
## Azure OpenAI Service Edition
## Learning Objectives
By the end of this notebook, you will be able to:
- Understand what sessions are and why they are essential
- Create stateful conversations that remember context
- Manage conversation sessions with Azure AI
- Build natural, flowing interactions with a travel assistant
- Learn when to create new sessions vs. reuse existing ones

## Key Concepts

### What is a Session?
**Session (AgentSession)** is a conversation history that maintains context across multiple queries.

**Without a Session:**
```
User: "I want to go to Japan"
Agent: "Great! Japan has many attractions..."

User: "What's the weather like there?"
Agent: "Which location would you like weather info for?"  Forgot about Japan!
```

**With a Session:**
```
User: "I want to go to Japan"
Agent: "Great! Japan has many attractions..."

User: "What's the weather like there?"
Agent: "Let me check the weather in Japan for you..."  Remembers!
```

### Types of Sessions

1. **Automatic Sessions** (Stateless)
   - A new session is created for each `run()` call
   - No memory between queries
   - Good for: one-off questions

2. **Persistent Sessions** (Stateful)
   - The same session is reused across multiple calls
   - Maintains complete conversation history
   - Good for: multi-turn conversations

---


## Step 1: Setup and Imports


In [ ]:
import asyncio
import os
from random import randint, choice
from typing import Annotated

from agent_framework import Agent, AgentSession
from pydantic import Field
from dotenv import load_dotenv

load_dotenv(override=True)
print("✅ Imports successful!")


In [ ]:
import os
from agent_framework.azure import AzureOpenAIChatClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Select Authentication Method ===
# True: API Key auth, False: DefaultAzureCredential auth
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 Auth method: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework auto-reads AZURE_OPENAI_API_KEY from env vars
    # and prioritizes it over credential, so explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Auth method: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: Since we use Agent with `async with`, we create a new client for each execution.
def create_azure_openai_chat_client():
    return AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment
    )

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")


## Setting Up a Tracer with OpenTelemetry
Using an OpenTelemetry tracer is helpful for debugging multi-agent scenarios.
Since `setup_observability` is not provided in this environment's Agent Framework,
we manually set up the OpenTelemetry `TracerProvider` (including Exporter/Processor) and enable instrumentation with `enable_instrumentation()`.
Here we use OTLP gRPC (e.g., Jaeger / OpenTelemetry Collector on port `4317`) as the trace destination.

Jaeger UI: [http://localhost:16686](http://localhost:16686)


In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# Configure via environment variables (Agent Framework reads standard OTEL_* vars)
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
# Since Jaeger only supports traces, set the trace-specific TRACES_ENDPOINT
# instead of the common ENDPOINT to avoid metrics/logs send errors
os.environ.setdefault("OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")

# Set enable_sensitive_data=True to enable collection of sensitive data (OpenAI IN/OUT data)
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")


## Step 2: Define Travel Tools

Let's create the tools that the agent will use throughout the conversation.


In [ ]:
def get_weather(
    location: Annotated[str, Field(description="City or country name")],
) -> str:
    """Returns the current weather for the specified location."""
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Windy"]
    temp = randint(15, 32)
    condition = choice(conditions)
    return f"Weather in {location}: {condition}, {temp}°C"


def get_attractions(
    location: Annotated[str, Field(description="City or country name")],
) -> str:
    """Returns the main tourist attractions for the destination."""
    # Mock data for tourist attractions
    attractions = {
        "Japan": "Tokyo Tower, Mount Fuji, Kyoto Temples, Osaka Castle",
        "Paris": "Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, Arc de Triomphe",
        "London": "Big Ben, Tower of London, British Museum, Buckingham Palace",
        "Barcelona": "Sagrada Familia, Park Guell, La Rambla, Gothic Quarter",
    }

    for city, attrs in attractions.items():
        if city.lower() in location.lower():
            return f"Top attractions in {location}: {attrs}"

    return f"Popular attractions in {location}: Historical sites, Museums, Local markets, Parks"


def estimate_budget(
    destination: Annotated[str, Field(description="Destination city or country")],
    days: Annotated[int, Field(description="Number of days")],
) -> str:
    """Estimates the travel budget based on destination and number of days."""
    daily_costs = {
        "Japan": 150,
        "Paris": 180,
        "London": 200,
        "Barcelona": 120,
        "Thailand": 60,
    }

    cost_per_day = daily_costs.get(destination, 100)
    total = cost_per_day * days

    return f"Estimated budget for {days} days in {destination}: ${total} (${cost_per_day} per day)"


print("✅ Tool definitions complete")


## Step 3: The Problem - No Memory (Stateless)

Let's see what happens without a thread - each query is independent.


In [ ]:
async def example_without_threads():
    """
    Demonstrates the problem: the agent doesn't remember previous context.
    A new session is automatically created for each run() call.
    """
    print("=== Without Session (No Memory) ===")
    print("Each query is independent - the agent forgets previous conversations\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="StatelessTravelAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        # First query
        query1 = "I'm planning a trip to Japan next month."
        print(f"User: {query1}")
        response1 = await agent.run(query1)
        print(f"Agent: {response1.text}\n")

        # Second query - expecting the agent to remember Japan
        query2 = "What's the weather like there?"
        print(f"User: {query2}")
        response2 = await agent.run(query2)
        print(f"Agent: {response2.text}\n")

        print("❌ Notice: The agent doesn't know where \"there\" is!")
        print("   Each run() created a separate session.\n")


await example_without_threads()


## Step 4: The Solution - Persistent Threads

Now let's use **persistent threads** to maintain conversation context.


In [ ]:
async def example_with_session():
    """
    This shows the solution: using the same session maintains context.
    """
    print("=== With Persistent Session (With Memory) ===")
    print("The same session maintains context across multiple queries\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="StatefulTravelAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        # Create a new session that we'll reuse
        session = agent.create_session()

        # First query
        query1 = "I'm planning a trip to Kyoto next month."
        print(f"User: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"Agent: {response1.text}\n")

        # Second query - agent remembers Japan!
        query2 = "What's the weather like there?"
        print(f"User: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"Agent: {response2.text}\n")

        # Third query - agent remembers the whole conversation
        query3 = "What are the top attractions to visit?"
        print(f"User: {query3}")
        response3 = await agent.run(query3, session=session)
        print(f"Agent: {response3.text}\n")

        print("✅ Success: The agent remembers the entire conversation!")
        print(f"   Session ID: {session.session_id}\n")

await example_with_session()


## Step 5: A Natural Multi-Turn Planning Conversation

Let's have a realistic travel planning conversation with multiple interaction turns.


In [ ]:
async def planning_conversation():
    """
    A natural conversation where the agent helps plan a trip.
    """
    print("=== Natural Travel Planning Conversation ===")
    print("Context-aware multi-turn dialogue\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="TravelPlanningAgent",
            instructions="""
            You are a travel planning expert. Help the user plan their trip by:
            - Asking clarifying questions as needed
            - Remembering all details the user shares
            - Using tools to provide accurate information
            - Being enthusiastic and helpful
            """,
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()

        # Conversation flow
        queries = [
            "I have 2 weeks off in December and want to go somewhere warm.",
            "I'm considering Thailand or Barcelona. Which would you recommend?",
            "I've decided on Barcelona! What's the weather like in December?",
            "How much budget should I plan for 7 days?",
            "That works for me! What are the must-see attractions?",
        ]

        for i, query in enumerate(queries, 1):
            print(f"{'='*60}")
            print(f"Turn {i}")
            print(f"{'='*60}")
            print(f"User: {query}\n")

            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")

        print("✅ Conversation completed with full context maintained!")
        print(f"   Session ID: {session.session_id}\n")

await planning_conversation()


## Step 6: Multiple Conversations with Different Threads

You can manage multiple conversations simultaneously using different threads.


In [ ]:
async def multiple_conversations():
    """
    Manage multiple separate conversations with different sessions.
    """
    print("=== Multiple Conversations (Different Sessions) ===")
    print("Each session maintains its own context\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="MultiConversationAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, get_attractions],
        ) as agent,
    ):
        # Create two separate sessions for two different users/conversations
        alice_session = agent.create_session()
        bob_session = agent.create_session()

        # Alice's conversation about Paris
        print("🧑 Alice's Conversation:")
        alice_q1 = "I'm interested in visiting Paris."
        print(f"Alice: {alice_q1}")
        response = await agent.run(alice_q1, session=alice_session)
        print(f"Agent: {response.text[:100]}...\n")

        # Bob's conversation about London
        print("👨 Bob's Conversation:")
        bob_q1 = "I'm thinking about London for my next trip."
        print(f"Bob: {bob_q1}")
        response = await agent.run(bob_q1, session=bob_session)
        print(f"Agent: {response.text[:100]}...\n")

        # Continue Alice's conversation
        print("🧑 Alice continues:")
        alice_q2 = "What's the weather like there?"
        print(f"Alice: {alice_q2}")
        response = await agent.run(alice_q2, session=alice_session)
        print(f"Agent: {response.text}\n")

        # Continue Bob's conversation
        print("👨 Bob continues:")
        bob_q2 = "What attractions should I visit there?"
        print(f"Bob: {bob_q2}")
        response = await agent.run(bob_q2, session=bob_session)
        print(f"Agent: {response.text}\n")

        print("✅ Both conversations maintained separate context!")
        print(f"   Alice's Session ID: {alice_session.session_id}")
        print(f"   Bob's Session ID: {bob_session.session_id}\n")

await multiple_conversations()


## Step 7: Resuming Conversations (Two Approaches)

How to "resume a conversation later" depends on whether the **chat_client can manage threads on the service side**.

### (A) Resume with Foundry (Azure AI) Service-Managed Threads
- `thread.service_thread_id` is an ID that points to a **conversation thread stored on the service side (Foundry)**.
- If you save this ID to a database, you can resume later with `AgentThread(service_thread_id=...)`.
- The `thread_id` in `AzureAIAgentClient(..., thread_id=...)` means "the default thread ID (= conversation/thread ID) that this client uses".

### (B) Resume with AzureOpenAIChatClient Local History
- The Chat Completions-based `AzureOpenAIChatClient` typically does not rely on **service-managed threads like Foundry's thread_id**.
- In that case, save the result of `AgentThread.serialize()` (local conversation history) and restore it later with `agent.deserialize_thread(...)` to resume.

The following cells demonstrate (A) and (B) respectively.


In [ ]:
async def resume_conversation_foundry():
    """
    Example of resuming a conversation with local history using create_azure_openai_chat_client.
    """
    import json

    print("=== AzureOpenAIChatClient: Resume Conversation with Local History ===")

    if not (azure_openai_key and azure_openai_endpoint and api_version and azure_deployment):
        print("⚠️ Missing environment variables for Azure OpenAI.")
        print("   AZURE_OPENAI_API_KEY / AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_API_VERSION / AZURE_OPENAI_CHAT_DEPLOYMENT")
        return

    saved_session_state: dict | None = None

    print("\n--- Part 1: Start a conversation and save session state ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()
        query = "I want to visit Barcelona for 5 days. What would the budget be?"
        print(f"User: {query}")
        response = await agent.run(query, session=session)
        print(f"Agent: {response.text}\n")

        saved_session_state = session.to_dict()
        preview = json.dumps(saved_session_state, ensure_ascii=False)
        print("✅ Session state saved (showing a portion of the JSON)")
        print(preview[:300] + ("..." if len(preview) > 300 else ""))

    if not saved_session_state:
        print("⚠️ Failed to save session state.\n")
        return

    print("--- Part 2: Resume conversation from saved session state ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = AgentSession.from_dict(saved_session_state)
        query = "What's the weather like there?"
        print(f"User: {query}")
        response = await agent.run(query, session=session)
        print(f"Agent: {response.text}\n")

        print("✅ Successfully resumed conversation from local history!")

await resume_conversation_foundry()


In [ ]:
async def resume_conversation_azure_openai_local():
    """
    (B) Example of resuming a conversation by preserving local history with AzureOpenAIChatClient (Chat Completions).
    - What we save here is not a service_session_id but the AgentSession's serialized state (conversation history).
    - In a real app, you would save this state to a DB/Blob/Redis, etc.
    """
    import json

    print("=== (B) AzureOpenAIChatClient: Resume Conversation with Local History ===")

    saved_session_state: dict | None = None

    # Part 1: Start a conversation and save the serialized session state (local history)
    print("\n--- Part 1: Start a conversation and save session state ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()
        query = "I want to visit Barcelona for 5 days. What would the budget be?"
        print(f"User: {query}")
        response = await agent.run(query, session=session)
        print(f"Agent: {response.text}\n")

        saved_session_state = session.to_dict()
        preview = json.dumps(saved_session_state, ensure_ascii=False)

        with open("saved_session_state.json", "w", encoding="utf-8") as f:
            json.dump(saved_session_state, f, ensure_ascii=False, indent=2)

        print("✅ Session state saved (showing a portion of the JSON)")
        print(preview[:300] + ("..." if len(preview) > 300 else ""))
        print("   (In a real app, you would save this JSON to a database, etc.)\n")

    if not saved_session_state:
        print("⚠️ Failed to save session state.\n")
        return

    # Part 2: Resume later by deserializing the saved session state
    print("--- Part 2: Resume conversation from saved session state ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = AgentSession.from_dict(saved_session_state)
        query = "What's the weather like there?"
        print(f"User: {query}")
        response = await agent.run(query, session=session)
        print(f"Agent: {response.text}\n")

        print("✅ Successfully resumed conversation from local history!")
        print("   The agent remembered Barcelona from the previous session.\n")

await resume_conversation_azure_openai_local()


## Step 8: Inspecting Thread Messages

You can examine the conversation history stored in the thread.


In [ ]:
async def inspect_session():
    """
    Inspect the messages and history within a session.
    Note: With AzureOpenAIChatClient, sessions are treated as local history.
    """
    print("=== Inspecting Session Messages ===")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="InspectableAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather],
        ) as agent,
    ):
        session = agent.create_session()

        # Have a short conversation
        queries = ["What's the weather in Tokyo?", "How about London?", "Which has better weather?"]

        print("Conversing...\n")
        for i, query in enumerate(queries, 1):
            print(f"Query {i}: {query}")
            response = await agent.run(query, session=session)
            print(f"Response: {response.text[:100]}...\n")

        # Inspect the session
        print("=" * 60)
        print(f"Session ID: {session.session_id}")
        print(f"Service-Managed Session: {session.service_session_id is not None}")
        print(f"Session State: {session.state}")
        print("=" * 60)
        print("\nNote: In this configuration, conversation history can be saved by serializing the session state.")
        print("Use AgentSession.from_dict(...) to resume as needed.")

await inspect_session()


## 🗑 Agent Lifecycle Management in Microsoft Foundry

### Automatic Agent Cleanup

**Important:** In Microsoft Foundry, depending on how you create agents via the SDK, **agents are automatically deleted after execution completes.**

#### How It Works

```python
async with (
    AzureCliCredential() as credential,
    Agent(
        client=AzureAIAgentClient(credential=credential),
        name="TravelAgent",
        instructions="You are a travel assistant.",
    ) as agent,  # ← Agent is created here
):
    # Agent exists in Azure within this block
    response = await agent.run("Hello")
    
# ← Agent is deleted from Azure (when context exits)
```

**What happens:**
1. **Agent Creation** - When entering the `async with` block, the agent is created in Microsoft Foundry
2. **Agent Usage** - You can execute queries, create sessions, and use tools
3. **Agent Deletion** - When exiting the block, the agent definition is deleted from Azure
4. **Sessions Persist** - Session data remains in Azure (only the agent definition is deleted)

### What Gets Deleted vs. What Persists

| Resource | After Context Exit | Reason |
|----------|-------------------|--------|
| **Agent Definition** | ❌ Deleted | Reduces clutter and saves costs |
| **Agent Name** | ❌ Deleted | Agent resource is deleted |
| **Agent Instructions** | ❌ Deleted | Part of agent definition |
| **Agent Tools** | ❌ Deleted | Part of agent definition |
| **Session ID** | ✅ Persists | Conversation history is retained |
| **Session Messages** | ✅ Persists | Data is retained |
| **Model Deployment** | ✅ Persists | Shared resource |

### Benefits of Automatic Deletion

#### 1. **Cost Efficiency** 
```python
# Traditional approach (agents persist)
agent_id = "agent-123"  # Agent stays in Azure, potentially causing governance and maintenance issues

# Microsoft Foundry approach (ephemeral agents)
async with Agent(...) as agent:
    # Agent exists only when needed
    await agent.run(query)
# Agent deleted
```

**Benefit:** You only pay for compute while running, not for idle agent storage

#### 2. **Resource Management** 🧹
```python
# Problem with persistent agents:
# - Accumulation of stale/test agents
# - Namespace pollution
# - Manual cleanup required

# Microsoft Foundry solution:
async with Agent(name="TestAgent") as agent:
    # Create and test
    pass
# Automatic cleanup - no test agents left behind!
```

**Benefit:** No "agent sprawl" - your Azure workspace stays clean

#### 3. **Security and Compliance** 
```python
# Sensitive agent with PII access
async with Agent(
    name="CustomerDataAgent",
    instructions="Access customer PII for support purposes",
) as agent:
    # Agent exists temporarily
    response = await agent.run("Retrieve user information")
# Agent definition deleted, reducing attack surface
```

**Benefit:** Short-lived credentials, reduced security risk

#### 4. **Version Control and GitOps** 
```python
# Agent definition in code
async with Agent(
    name="V2TravelAgent",  # Version in code
    instructions="Updated instructions..."  # Source controlled
) as agent:
    await agent.run(query)
```

**Benefit:** Agent behavior is defined in code, not in Azure state

### Drawbacks and Tradeoffs

| Drawback | Mitigation |
|----------|------------|
| **Agent is recreated per execution** | Overhead is minimal (~100-300ms) |
| **Cannot view agents in portal** | View sessions/messages instead |
| **Requires code for definition** | Good for version control |
| **No long-running agents** | Use sessions for continuity |

### Alternative: Persistent Agents 

You can also make agents persistent:

```python
agent = create_agent(name="MyAgent")  # Agent remains on the platform
agent_id = agent.id  # "agent-abc123"

# Later (different session)
agent = get_agent(agent_id)  # Retrieve existing agent
response = agent.run("Hello")  # Use persistent agent
```

**When persistent agents make sense:**
- Agents with complex, evolving instructions
- Agents shared across multiple services
- Agents that learn/adapt over time
- Long-running background agents

**Microsoft Foundry Philosophy:**
- Agents are **code** (in your repo)
- Sessions are **data** (in Azure)
- Separate compute (ephemeral) from state (persistent)

### Microsoft Foundry Best Practices

#### 1. **Define Agents as Code**
```python
# Good: Agent configuration in code
def create_travel_agent(credential):
    return Agent(
        client=AzureAIAgentClient(credential=credential),
        name="TravelAgent",
        instructions="You are a helpful travel assistant...",
        tools=[get_weather, get_attractions]
    )

# Usage
async with create_travel_agent(credential) as agent:
    response = await agent.run(query, session=session)
```

#### 2. **Persist Session IDs, Not Agent IDs**
```python
# Save to database
session_data = {
    "user_id": user.id,
    "session_id": session.service_session_id,  # ✅ Save this
    # "agent_id": agent.id  # ❌ Don't do this (agent will be deleted)
}

# Restore later
async with create_travel_agent(credential) as agent:  # New agent
    session = AgentSession(service_session_id=session_data["session_id"])  # Same session
    response = await agent.run(query, session=session)
```

#### 3. **Use Context Managers**
```python
# ✅ Good: Automatic cleanup
async with Agent(...) as agent:
    await agent.run(query)

# ❌ Avoid: Manual management
agent = Agent(...)
await agent.run(query)
# Agent may not be properly cleaned up
```

### Comparison: Azure AI vs Traditional

| Aspect | Microsoft Foundry | Traditional Persistent |
|--------|------------------|------------------------|
| Agent Lifetime | Ephemeral (seconds) | Persistent (days/months) |
| Storage Cost | No agents | Ongoing agent cost |
| Cleanup | Automatic | Manual required |
| Version Control | In code (Git) | In platform (API) |
| Scaling | Unlimited agents | Quota limited |
| Session Persistence | ✅ Yes | ✅ Yes |
| Agent Evolution | Code updates | Platform updates |
| Multi-tenancy | Isolated per run | Shared agents |


## Understanding the Session Lifecycle

### Session Creation and Management

```python
# Option 1: Let sessions be created and managed automatically
response = await agent.run(query)  # New session each time

# Option 2: Create a session and reuse it (recommended)
session = agent.create_session()
response = await agent.run(query, session=session)  # Same session

# Option 3: Resume from a saved session ID
session = AgentSession(service_session_id=saved_id)
response = await agent.run(query, session=session)
```

### Session Storage in Azure

- Sessions are **stored in Microsoft Foundry**
- Session IDs are **persistent** across sessions
- You can **resume conversations** at any time
- Azure manages **cleanup and retention**

### When to Use Sessions

| Use Case | Session Strategy |
|----------|------------------|
| One-off questions | No session (automatic) |
| Chat conversations | Single persistent session |
| Multi-user apps | One session per user |
| Session-based | New session per session |
| Long-term assistant | Save and resume session IDs |


## 🔬 Hands-on Example: Agent Lifecycle Demo

Let's see ephemeral agent behavior in action:


In [ ]:
async def agent_lifecycle_demo():
    """
    Demonstrates session isolation and conversation resumption using AzureOpenAIChatClient.
    """
    print("=== Agent Lifecycle Demo (AOAI) ===\n")

    if not (azure_openai_key and azure_openai_endpoint and api_version and azure_deployment):
        print("⚠️ Missing environment variables for Azure OpenAI.")
        print("   AZURE_OPENAI_API_KEY / AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_API_VERSION / AZURE_OPENAI_CHAT_DEPLOYMENT")
        return

    saved_session_state: dict | None = None

    print("--- Session 1: Create agent → Converse → Save state ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="EphemeralAgent_Session1",
            instructions="You are a helpful assistant. Please remember the information the user tells you.",
            tools=[get_weather],
        ) as agent,
    ):
        print(f"✅ Agent created: {agent.name}")

        session = agent.create_session()
        query1 = "My favorite city is Barcelona. Please remember that."
        print(f"User: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"Agent: {response1.text}\n")

        saved_session_state = session.to_dict()
        print("✅ Session state saved")

    if not saved_session_state:
        print("⚠️ Could not save session state.")
        return

    print("--- Session 2: Resume conversation with a new agent ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="EphemeralAgent_Session2",
            instructions="You are a helpful assistant. Please remember the information the user tells you.",
            tools=[get_weather],
        ) as agent,
    ):
        print(f"✅ New agent created: {agent.name}\n")

        session = AgentSession.from_dict(saved_session_state)

        query2 = "What was my favorite city?"
        print(f"User: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"Agent: {response2.text}\n")

        query3 = "What's the weather like in that city?"
        print(f"User: {query3}")
        response3 = await agent.run(query3, session=session)
        print(f"Agent: {response3.text}\n")

    print("=" * 60)
    print("Key Insights:")
    print("=" * 60)
    print("1. Agents are created per session")
    print("2. Use session state save/restore for conversation continuity")
    print("3. Different agents can continue the same conversation")
    print("=" * 60)

await agent_lifecycle_demo()


## Resource Scope / Lifetime
The `async with (...)` above may look "ugly," but what it does is straightforward: it reliably creates and destroys multiple "resources with cleanup" in a nested fashion. The key point is that **scope** (valid range) and **lifetime** (when created and when closed/destroyed) are strictly determined by `async with`.

Within this single block, there are at least the following items with different lifetimes:

- `credential` (authentication handle)
  - Lifetime: Valid only within the `async with DefaultAzureCredential()` block
  - On exit: `credential` is closed, and internal HTTP connections may also be closed

- The client returned by `create_azure_openai_chat_client()` (`AzureOpenAIChatClient`)
  - Lifetime: Only used within `Agent(... ) as agent` (ephemeral)
  - On exit: Client usage ends along with the Agent's termination

- `agent` (the short-lived agent created on the Foundry side)
  - Lifetime: Only within the `async with ... as agent:` block
  - On exit: **The agent definition on the service side is deleted** the moment you exit the block (= ephemeral)

- `session` (conversation session)
  - Lifetime: Two possibilities
    - **Service-managed session**: A `session.service_session_id` is issued, and you can save the ID to restore it later (the session persists even after the agent is deleted)
    - Otherwise: Held locally only, and won't persist unless the state is saved before the process ends

In other words, `async with` enforces the design principle of "agents are short-lived, sessions are persistent (potentially)."

## Why It Looks "Ugly" (Why It's Hard to Read)
- A single `async with` line packs in **3 levels of lifecycle** (credential / client / agent)
- Additionally, `.as_agent(...)` is a two-step process of "generating a context manager for the agent from the client," making the creation and destruction boundaries hard to see


## Key Takeaways

### Benefits of Sessions
1. **Context Awareness** - Agent remembers previous interactions
2. **Natural Conversations** - Users don't need to repeat information
3. **Better UX** - Feels like talking to a human
4. **Stateful Interactions** - Builds on previous responses

### Best Practices
1. **Use sessions for conversations** - Always use for multi-turn dialogues
2. **Save session IDs** - Store in user database
3. **One session per conversation** - Don't mix different topics
4. **Clean up old sessions** - Delete when conversation ends (Azure helps with this)

### Common Patterns

**Web Chat Application:**
```python
# Save session_id in user session
if not session_store.get('session_id'):
    session = agent.create_session()
    session_store['session_id'] = session.service_session_id
else:
    session = AgentSession(service_session_id=session_store['session_id'])
```

**Customer Support:**
```python
# One session per support ticket
session_id = ticket.get_session_id()
session = AgentSession(service_session_id=session_id)
```


## Practice Exercises

1. **Create a Booking Conversation** - Plan a complete trip from start to booking
2. **Destination Comparison** - Have the agent remember and compare multiple cities
3. **Build a Preference Profile** - Have the agent learn user preferences through conversation
4. **Implement Session Management** - Save/load session IDs from a dictionary


In [ ]:
# Exercise: Create a conversation where the agent helps compare 3 destinations
# and remembers all the details discussed about each one

async def compare_destinations_exercise():
    """
    Your turn! Create a conversation that:
    1. Discusses Paris (weather, attractions, budget)
    2. Discusses Tokyo (same info)
    3. Discusses Barcelona (same info)
    4. Asks agent to compare all three and recommend
    
    The agent should remember ALL details from the conversation!
    """
    # Your code here
    pass

# Uncomment to test your solution
# await compare_destinations_exercise()


## Next Steps

Great! You now understand how to create stateful, context-aware conversations.

However, agents still have limitations:
- ❌ They don't remember user preferences across sessions
- ❌ They can't store long-term knowledge about users
- ❌ Session conversation history can become very long

**In Tutorial 03: Context and Memory**, you will learn:
- How to add persistent memory to agents
- How to save user preferences and profiles
- How to use context providers for smarter agents
- How to manage conversation context efficiently

---

### Quick Reference

**Create a Persistent Session:**
```python
session = agent.create_session()
response = await agent.run(query, session=session)
```

**Save and Resume:**
```python
# Save
session_id = session.service_session_id

# Resume
session = AgentSession(service_session_id=session_id)
```

**Multiple Conversations:**
```python
alice_session = agent.create_session()
bob_session = agent.create_session()
```
